In [ ]:
# =============================================================================
# EshMagan v2 — FIRE RISK, DETECTION, REPORTING & EVACUATION ROUTING
# =============================================================================

In [ ]:
# ===========================================================================
# CELL 1 — Install required libraries
# ===========================================================================

!pip install -q requests beautifulsoup4 scikit-learn matplotlib seaborn pandas
!pip install -q numpy tqdm pytz pdfplumber smbus2
!pip install -q tensorflow
!pip install -q shapely geopandas networkx osmnx pyproj

print("All libraries assumed installed. Starting imports …")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.4/68.4 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 66.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 57.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 104.4/104.4 kB 5.3 MB/s eta 0:00:00
All libraries assumed installed. Starting imports …


In [ ]:
# ===========================================================================
# CELL 2 — Imports
# ===========================================================================

import os, json, time, math, warnings, io, re
import requests
import pdfplumber
from datetime import datetime, date, timedelta
from pathlib import Path
from urllib.parse import urljoin

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import pytz
from bs4 import BeautifulSoup

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, models, callbacks, regularizers

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report,
    roc_auc_score, roc_curve, precision_recall_curve,
    f1_score, precision_score, recall_score, accuracy_score,
)
from sklearn.utils.class_weight import compute_class_weight

# Geospatial stack
from shapely.geometry import (
    Point, LineString, Polygon, MultiPolygon, mapping, shape
)
from shapely.ops import unary_union
import shapely.wkt as swkt
import geopandas as gpd
import networkx as nx
import osmnx as ox
from pyproj import Transformer, CRS

warnings.filterwarnings("ignore")
tf.get_logger().setLevel("ERROR")

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs found : {tf.config.list_physical_devices('GPU')}")

TensorFlow : 2.20.0
GPUs found : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
# ===========================================================================
# CELL 3 — UOB Parameter Weights
# ===========================================================================
# SOURCE: "UOB Fire Ignition Parameter Weights", IOE, University of Balamand.
# All six weights must sum to exactly 1.0.

UOB_WEIGHTS = {
    "humidity": {
        "weight": 0.34, "inverse": True,
        "label": "Relative humidity (%)", "norm_min": 0, "norm_max": 100,
    },
    "rainfall": {
        "weight": 0.28, "inverse": True,
        "label": "Rainfall (mm)", "norm_min": 0, "norm_max": 20,
    },
    "temperature": {
        "weight": 0.17, "inverse": False,
        "label": "Temperature (°C)", "norm_min": -5, "norm_max": 50,
    },
    "wind_speed": {
        "weight": 0.11, "inverse": False,
        "label": "Wind speed (km/h)", "norm_min": 0, "norm_max": 100,
    },
    "pressure": {
        "weight": 0.06, "inverse": False,
        "label": "Pressure (hPa)", "norm_min": 980, "norm_max": 1030,
    },
    "wind_dir": {
        "weight": 0.04, "inverse": False,
        "label": "Wind direction (°)", "norm_min": 0, "norm_max": 360,
    },
}

SENSOR_ORDER = list(UOB_WEIGHTS.keys())
N_SENSORS = len(SENSOR_ORDER)

total = sum(v["weight"] for v in UOB_WEIGHTS.values())
print(f"UOB weights loaded. Sum = {total*100:.0f}%")

UOB weights loaded. Sum = 100%


In [ ]:
# ===========================================================================
# CELL 4 — Fire Weather Index (FWI) functions
# ===========================================================================

def normalise_param(key: str, value: float) -> float:
    """Min-max normalise one weather reading to [0, 1]. Flip if inverse."""
    cfg = UOB_WEIGHTS[key]
    if key == "wind_dir":
        return 0.5  # No terrain map yet; fixed neutral score
    lo, hi = cfg["norm_min"], cfg["norm_max"]
    norm = np.clip((value - lo) / (hi - lo + 1e-9), 0.0, 1.0)
    return float(1.0 - norm if cfg["inverse"] else norm)


def compute_uob_fwi(weather: dict) -> dict:
    """
    Compute the UOB-weighted Fire Weather Index.

    Returns
    -------
    fwi_score       : 0–100 risk score
    compound_flag   : True when hot + dry + no rain simultaneously
    contributions   : % contribution per parameter
    weather_vector  : float32 array of shape (6,) for the neural network
    """
    contributions = {}
    for key, cfg in UOB_WEIGHTS.items():
        if key not in weather:
            contributions[key] = 0.0
            continue
        norm = normalise_param(key, weather[key])
        norm_nl = norm ** 0.80          # Slight concave boost for mid-range readings
        contributions[key] = cfg["weight"] * norm_nl

    base_score = sum(contributions.values())

    compound = (
        weather.get("humidity", 50) < 30
        and weather.get("rainfall", 1) == 0
        and weather.get("temperature", 20) > 30
    )
    multiplier = 1.35 if compound else 1.0
    final_score = min(base_score * multiplier * 100, 99.9)

    weather_vector = np.array(
        [normalise_param(k, weather.get(k, 0)) for k in SENSOR_ORDER],
        dtype=np.float32,
    )

    return {
        "fwi_score": round(final_score, 2),
        "compound_flag": compound,
        "contributions": {k: round(v * 100, 2) for k, v in contributions.items()},
        "weather_vector": weather_vector,
    }


def fwi_class(score: float) -> str:
    """Map a numeric FWI score to a human-readable danger class."""
    if score < 15: return "Very Low"
    if score < 30: return "Low"
    if score < 45: return "Moderate"
    if score < 60: return "High"
    if score < 75: return "Very High"
    return "Extreme"


def predict_fire_behavior(weather: dict, fwi: float) -> dict:
    """
    Estimate fire spread behaviour using a simplified Rothermel model
    calibrated for Mediterranean scrub and pine (dominant fuel at UOB).
    """
    ws  = weather.get("wind_speed",   10)
    wd  = weather.get("wind_dir",    180)
    rh  = weather.get("humidity",     50)
    tmp = weather.get("temperature",  20)

    rate_of_spread = max(0, (ws * 0.45 + (100 - rh) * 0.30 + (tmp - 15) * 0.25) * 0.7)
    flame_low  = round(0.4 + rate_of_spread * 0.07, 1)
    flame_high = round(1.2 + rate_of_spread * 0.13, 1)

    dirs = ["N","NNE","NE","ENE","E","ESE","SE","SSE",
            "S","SSW","SW","WSW","W","WNW","NW","NNW"]
    spread_dir = dirs[round(wd / 22.5) % 16]

    crown_risk = (
        "High"     if rh < 25 and ws > 30 else
        "Moderate" if rh < 35 and ws > 15 else
        "Low"
    )

    return {
        "rate_of_spread_m_per_min": round(rate_of_spread, 1),
        "spread_direction":         spread_dir,
        "wind_direction_deg":       wd,
        "flame_length_m":           f"{flame_low}–{flame_high}",
        "crown_fire_risk":          crown_risk,
        "spotting_distance_km":     round(ws * 0.018, 2),
        "ember_transport_risk":     (
            "High"     if ws > 25 else
            "Moderate" if ws > 12 else
            "Low"
        ),
    }

In [ ]:
# ===========================================================================
# CELL 5 — MLX90640 thermal sensor analysis
# ===========================================================================

THERMAL_HOT_THRESHOLD_C = 43.0
THERMAL_CRITICAL_MAX_C  = 50.0
MIN_HOTSPOT_AREA        = 2
MIN_HOTSPOT_FRACTION    = 0.003
_NEIGHBOURS_8 = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]


def _connected_hotspot_areas(mask: np.ndarray) -> list:
    """Iterative DFS connected-component analysis on a boolean hot-pixel mask."""
    rows, cols = mask.shape
    visited = np.zeros_like(mask, dtype=bool)
    areas = []
    for r in range(rows):
        for c in range(cols):
            if not mask[r, c] or visited[r, c]:
                continue
            stack = [(r, c)]
            visited[r, c] = True
            area = 0
            while stack:
                cr, cc = stack.pop()
                area += 1
                for dr, dc in _NEIGHBOURS_8:
                    nr, nc = cr + dr, cc + dc
                    if 0 <= nr < rows and 0 <= nc < cols:
                        if mask[nr, nc] and not visited[nr, nc]:
                            visited[nr, nc] = True
                            stack.append((nr, nc))
            areas.append(area)
    return areas


def analyse_ir_frame_mlx90640(frame: np.ndarray) -> dict:
    """Convert a raw 24×32 thermal frame into structured fire evidence."""
    frame = np.asarray(frame, dtype=np.float32)
    if frame.shape != (24, 32):
        raise ValueError(f"Expected (24, 32) frame, got {frame.shape}")

    max_temp  = float(np.max(frame))
    mean_temp = float(np.mean(frame))

    hot_mask         = frame >= THERMAL_HOT_THRESHOLD_C
    hot_pixel_count  = int(np.sum(hot_mask))
    hotspot_fraction = hot_pixel_count / frame.size

    hotspot_areas        = _connected_hotspot_areas(hot_mask)
    hotspot_count        = len(hotspot_areas)
    largest_hotspot_area = max(hotspot_areas) if hotspot_areas else 0
    hotspot_mean_temp    = float(np.mean(frame[hot_mask])) if hot_pixel_count > 0 else 0.0

    conf_fraction = min(hotspot_fraction / 0.05, 1.0) * 0.30
    conf_area     = min(largest_hotspot_area / 20.0, 1.0) * 0.35
    conf_max      = min(max(max_temp - THERMAL_CRITICAL_MAX_C, 0.0) / 40.0, 1.0) * 0.35
    ir_confidence = round(conf_fraction + conf_area + conf_max, 4)

    fire_detected = (
        (largest_hotspot_area >= MIN_HOTSPOT_AREA and max_temp >= THERMAL_CRITICAL_MAX_C)
        or (hotspot_fraction >= MIN_HOTSPOT_FRACTION and max_temp >= THERMAL_HOT_THRESHOLD_C + 10)
        or (max_temp >= 100.0 and hot_pixel_count >= 3)
    )

    return {
        "fire_detected":         bool(fire_detected),
        "ir_confidence":         ir_confidence,
        "max_temp_c":            round(max_temp, 1),
        "mean_temp_c":           round(mean_temp, 1),
        "hot_pixel_count":       hot_pixel_count,
        "hotspot_count":         hotspot_count,
        "largest_hotspot_area":  int(largest_hotspot_area),
        "hotspot_fraction":      round(hotspot_fraction, 4),
        "hotspot_mean_temp_c":   round(hotspot_mean_temp, 1),
        "frame_shape":           frame.shape,
    }

In [ ]:
# ===========================================================================
# CELL 6 — Runtime feature schema
# ===========================================================================

FEATURE_NAMES = [
    "humidity_norm",
    "rainfall_norm",
    "temperature_norm",
    "wind_speed_norm",
    "pressure_norm",
    "wind_dir_neutral",
    "fwi_score_norm",
    "max_temp_norm",
    "hotspot_count_norm",
    "largest_hotspot_area_norm",
    "hotspot_fraction_norm",
    "thermal_confidence",
    "thermal_delta_norm",
]

N_FEATURES = len(FEATURE_NAMES)

In [ ]:
# ===========================================================================
# CELL 7 — Runtime Checkpoint Path
# ===========================================================================

CKPT_DIR  = Path("/content/firewatch_checkpoints")
CKPT_DIR.mkdir(exist_ok=True)
CKPT_PATH = str(CKPT_DIR / "best_model_v2.keras")

In [ ]:
# ===========================================================================
# CELL 8A — Runtime model metadata and TensorFlow Lite paths
# ===========================================================================

DEPLOY_DIR = Path("/content/firewatch_deploy")
DEPLOY_DIR.mkdir(exist_ok=True)

MODEL_METADATA_PATH = DEPLOY_DIR / "model_metadata_v2.json"

# TensorFlow Lite runtime model.
# The .keras checkpoint is still used only as the source if the .tflite file
# does not exist yet.
TFLITE_MODEL_PATH = str(CKPT_DIR / "firewatch_model_v2.tflite")

In [ ]:
# ===========================================================================
# CELL 8B — Train/export model if runtime assets are missing
# ===========================================================================
# This cell recreates the runtime assets:
# - /content/firewatch_checkpoints/best_model_v2.keras
# - /content/firewatch_deploy/model_metadata_v2.json


N_SYNTHETIC_SAMPLES = 15000
SEED = 42

rng = np.random.default_rng(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)


def generate_weather_sample(label: int) -> dict:
    """
    Produce one synthetic weather dict.
    Fire and non-fire samples overlap so the model learns ambiguity.
    """
    if label == 1:
        return {
            "humidity": float(rng.uniform(10, 50)),
            "rainfall": 0.0 if rng.random() > 0.12 else float(rng.uniform(0, 2.5)),
            "temperature": float(rng.uniform(22, 45)),
            "wind_speed": float(rng.uniform(4, 55)),
            "pressure": float(rng.uniform(1000, 1026)),
            "wind_dir": float(rng.uniform(0, 360)),
        }

    return {
        "humidity": float(rng.uniform(30, 95)),
        "rainfall": float(rng.uniform(0, 14)),
        "temperature": float(rng.uniform(5, 34)),
        "wind_speed": float(rng.uniform(0, 28)),
        "pressure": float(rng.uniform(1004, 1030)),
        "wind_dir": float(rng.uniform(0, 360)),
    }


def generate_thermal_features(label: int) -> dict:
    """
    Produce synthetic thermal features matching the 13-feature schema.
    """
    if label == 1:
        max_temp = float(rng.uniform(68, 135))
        mean_temp = float(rng.uniform(28, 45))

        return {
            "max_temp_c": max_temp,
            "mean_temp_c": mean_temp,
            "hotspot_count": int(rng.integers(1, 6)),
            "largest_hotspot_area": int(rng.integers(4, 40)),
            "hotspot_fraction": float(rng.beta(5, 2) * 0.08 + 0.005),
            "thermal_confidence": float(rng.beta(5, 2)),
        }

    max_temp = float(rng.uniform(22, 82))
    mean_temp = float(rng.uniform(18, 36))

    return {
        "max_temp_c": max_temp,
        "mean_temp_c": mean_temp,
        "hotspot_count": int(rng.integers(0, 3)),
        "largest_hotspot_area": int(rng.integers(0, 9)),
        "hotspot_fraction": float(rng.beta(2, 5) * 0.025),
        "thermal_confidence": float(rng.beta(2, 5)),
    }


def build_sensor_fusion_dataset(n_samples: int):
    """
    Build X/y training arrays using the same 13 features as runtime inference.
    """
    X, y = [], []

    for _ in range(n_samples):
        label = int(rng.random() > 0.5)

        weather = generate_weather_sample(label)
        fwi_result = compute_uob_fwi(weather)
        thermal = generate_thermal_features(label)

        row = list(fwi_result["weather_vector"])
        row.append(fwi_result["fwi_score"] / 100.0)
        row.append(np.clip((thermal["max_temp_c"] - 20.0) / 130.0, 0.0, 1.0))
        row.append(np.clip(thermal["hotspot_count"] / 5.0, 0.0, 1.0))
        row.append(np.clip(thermal["largest_hotspot_area"] / 50.0, 0.0, 1.0))
        row.append(np.clip(thermal["hotspot_fraction"] / 0.15, 0.0, 1.0))
        row.append(np.clip(thermal["thermal_confidence"], 0.0, 1.0))

        delta = thermal["max_temp_c"] - thermal["mean_temp_c"]
        row.append(np.clip(delta / 100.0, 0.0, 1.0))

        noise = rng.normal(0, 0.018, size=len(row))
        row = np.clip(np.array(row) + noise, 0.0, 1.0)

        X.append(row)
        y.append(label)

    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32)


def build_sensor_fusion_model(input_dim: int) -> keras.Model:
    """
    Same model architecture from the old AI notebook.
    """
    reg = regularizers.l2(0.001)

    inputs = layers.Input(shape=(input_dim,), name="sensor_fusion_input")

    x = layers.Dense(128, activation="relu", kernel_regularizer=reg, name="dense_1")(inputs)
    x = layers.BatchNormalization(name="bn_1")(x)
    x = layers.Dropout(0.35, name="drop_1")(x)

    x = layers.Dense(64, activation="relu", kernel_regularizer=reg, name="dense_2")(x)
    x = layers.BatchNormalization(name="bn_2")(x)
    x = layers.Dropout(0.25, name="drop_2")(x)

    x = layers.Dense(32, activation="relu", kernel_regularizer=reg, name="dense_3")(x)
    x = layers.BatchNormalization(name="bn_3")(x)
    x = layers.Dropout(0.15, name="drop_3")(x)

    x = layers.Dense(16, activation="relu", kernel_regularizer=reg, name="dense_4")(x)

    outputs = layers.Dense(1, activation="sigmoid", name="fire_probability")(x)

    return models.Model(
        inputs=inputs,
        outputs=outputs,
        name="EshMagan_UOB_Sensor_Fusion_NN_v2",
    )


def save_model_metadata(best_threshold: float, feature_names: list, ckpt_path: str):
    """
    Save all runtime metadata needed by the deployed inference service.
    """
    payload = {
        "model_name": "UOB_Sensor_Fusion_TFLite_v2",
        "checkpoint_path": ckpt_path,
        "best_threshold": float(best_threshold),
        "feature_names": feature_names,
        "n_features": len(feature_names),
        "saved_at": datetime.now().isoformat(),
        "thermal_hot_threshold_c": THERMAL_HOT_THRESHOLD_C,
        "thermal_critical_max_c": THERMAL_CRITICAL_MAX_C,
        "min_hotspot_area": MIN_HOTSPOT_AREA,
        "min_hotspot_fraction": MIN_HOTSPOT_FRACTION,
    }

    with open(MODEL_METADATA_PATH, "w", encoding="utf-8") as file:
        json.dump(payload, file, indent=2)

    print(f"Model metadata saved to: {MODEL_METADATA_PATH}")


def train_and_export_runtime_assets_if_missing(force_retrain: bool = False):
    """
    Train and export runtime assets only when missing.

    Creates:
    - best_model_v2.keras
    - model_metadata_v2.json
    """
    metadata_exists = Path(MODEL_METADATA_PATH).exists()
    checkpoint_exists = Path(CKPT_PATH).exists()

    if metadata_exists and checkpoint_exists and not force_retrain:
        print("Runtime assets already exist. Skipping training.")
        print(f"Metadata: {MODEL_METADATA_PATH}")
        print(f"Checkpoint: {CKPT_PATH}")
        return

    print("Runtime assets missing. Training model now...")

    X, y = build_sensor_fusion_dataset(N_SYNTHETIC_SAMPLES)

    print(f"Dataset: {X.shape[0]:,} samples × {X.shape[1]} features")
    print(f"Fire rate: {y.mean() * 100:.1f}%")

    idx = np.arange(len(y))

    idx_train, idx_temp = train_test_split(
        idx,
        test_size=0.20,
        stratify=y,
        random_state=SEED,
    )

    idx_val, idx_test = train_test_split(
        idx_temp,
        test_size=0.50,
        stratify=y[idx_temp],
        random_state=SEED,
    )

    X_train, y_train = X[idx_train], y[idx_train]
    X_val, y_val = X[idx_val], y[idx_val]
    X_test, y_test = X[idx_test], y[idx_test]

    model = build_sensor_fusion_model(N_FEATURES)

    class_weights = compute_class_weight(
        "balanced",
        classes=np.array([0, 1]),
        y=y_train,
    )

    class_weight_dict = {
        0: float(class_weights[0]),
        1: float(class_weights[1]),
    }

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            "accuracy",
            keras.metrics.Precision(name="precision"),
            keras.metrics.Recall(name="recall"),
            keras.metrics.AUC(name="auc"),
        ],
    )

    cb_list = [
        callbacks.ModelCheckpoint(
            filepath=CKPT_PATH,
            monitor="val_auc",
            mode="max",
            save_best_only=True,
            verbose=1,
        ),
        callbacks.EarlyStopping(
            monitor="val_auc",
            patience=10,
            restore_best_weights=True,
            mode="max",
            verbose=1,
        ),
        callbacks.ReduceLROnPlateau(
            monitor="val_loss",
            factor=0.4,
            patience=4,
            min_lr=1e-6,
            verbose=1,
        ),
        callbacks.TerminateOnNaN(),
    ]

    history = model.fit(
        X_train,
        y_train,
        validation_data=(X_val, y_val),
        epochs=25,
        batch_size=64,
        class_weight=class_weight_dict,
        callbacks=cb_list,
        verbose=1,
    )

    val_prob = model.predict(X_val, verbose=0).ravel()

    best_threshold = 0.5
    best_f1 = -1.0

    for threshold in np.linspace(0.1, 0.9, 81):
        score = f1_score(y_val, (val_prob >= threshold).astype(int))

        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)

    y_prob = model.predict(X_test, verbose=0).ravel()
    y_pred = (y_prob >= best_threshold).astype(int)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)

    print("=" * 72)
    print("Training finished.")
    print(f"Best threshold: {best_threshold:.4f}")
    print(f"Accuracy      : {acc:.4f}")
    print(f"Precision     : {prec:.4f}")
    print(f"Recall        : {rec:.4f}")
    print(f"F1            : {f1:.4f}")
    print(f"ROC-AUC       : {auc:.4f}")
    print("=" * 72)

    # Make sure the final/best model exists at CKPT_PATH.
    if not Path(CKPT_PATH).exists():
        model.save(CKPT_PATH)

    save_model_metadata(
        best_threshold=best_threshold,
        feature_names=FEATURE_NAMES,
        ckpt_path=CKPT_PATH,
    )


train_and_export_runtime_assets_if_missing(force_retrain=False)

Runtime assets missing. Training model now...
Dataset: 15,000 samples × 13 features
Fire rate: 50.2%
Epoch 1/25
188/188 ━━━━━━━━━━━━━━━━━━━━ 0s 62ms/step - accuracy: 0.9173 - auc: 0.9616 - loss: 0.3828 - precision: 0.9414 - recall: 0.8882
Epoch 1: val_auc improved from None to 1.00000, saving model to /content/firewatch_checkpoints/best_model_v2.keras

Epoch 1: finished saving model to /content/firewatch_checkpoints/best_model_v2.keras
188/188 ━━━━━━━━━━━━━━━━━━━━ 30s 85ms/step - accuracy: 0.9765 - auc: 0.9980 - loss: 0.2382 - precision: 0.9855 - recall: 0.9675 - val_accuracy: 0.9907 - val_auc: 1.0000 - val_loss: 0.2097 - val_precision: 0.9818 - val_recall: 1.0000 - learning_rate: 0.0010
Epoch 2/25
176/188 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - accuracy: 0.9987 - auc: 1.0000 - loss: 0.1333 - precision: 0.9982 - recall: 0.9992
Epoch 2: val_auc improved from 1.00000 to 1.00000, saving model to /content/firewatch_checkpoints/best_model_v2.keras

Epoch 2: finished saving model to /content/fire

In [ ]:
# ===========================================================================
# CELL 9A — Live feature vector builder + TensorFlow Lite prediction pipeline
# ===========================================================================

RUNTIME_MODEL = None
RUNTIME_THRESHOLD = None
RUNTIME_METADATA = None


def convert_keras_to_tflite():
    """
    Convert the saved Keras checkpoint into TensorFlow Lite format.

    This lets the runtime inference use TensorFlow Lite while still allowing
    the project to keep the trained .keras checkpoint as the source model.
    """
    checkpoint_path = Path(CKPT_PATH)

    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Missing Keras checkpoint file: {CKPT_PATH}. "
            "Cannot create TensorFlow Lite model."
        )

    keras_model = keras.models.load_model(CKPT_PATH)

    converter = tf.lite.TFLiteConverter.from_keras_model(keras_model)
    converter.optimizations = [tf.lite.Optimize.DEFAULT]

    tflite_model = converter.convert()

    with open(TFLITE_MODEL_PATH, "wb") as f:
        f.write(tflite_model)

    print("TensorFlow Lite model created.")
    print(f"  Source Keras model : {CKPT_PATH}")
    print(f"  TFLite model       : {TFLITE_MODEL_PATH}")


def load_runtime_inference_assets():
    """
    Load the TensorFlow Lite model and metadata for API inference.

    Runtime inference uses TensorFlow Lite.
    The .keras checkpoint is only used to generate the .tflite file if needed.
    """
    global RUNTIME_MODEL, RUNTIME_THRESHOLD, RUNTIME_METADATA

    metadata_path = Path(MODEL_METADATA_PATH)
    tflite_path = Path(TFLITE_MODEL_PATH)

    if not metadata_path.exists():
        raise FileNotFoundError(
            f"Missing model metadata file: {MODEL_METADATA_PATH}. "
            "Run/export the trained model metadata first."
        )

    if not tflite_path.exists():
        print("TensorFlow Lite model not found. Creating it from Keras checkpoint...")
        convert_keras_to_tflite()

    with open(metadata_path, "r", encoding="utf-8") as f:
        RUNTIME_METADATA = json.load(f)

    interpreter = tf.lite.Interpreter(model_path=TFLITE_MODEL_PATH)
    interpreter.allocate_tensors()

    RUNTIME_MODEL = interpreter
    RUNTIME_THRESHOLD = float(RUNTIME_METADATA["best_threshold"])

    print("Runtime TensorFlow Lite inference assets loaded.")
    print(f"  Model     : {TFLITE_MODEL_PATH}")
    print(f"  Threshold : {RUNTIME_THRESHOLD:.4f}")


def predict_with_tflite(interpreter, live_features: np.ndarray) -> float:
    """
    Run one prediction using the TensorFlow Lite interpreter.
    """
    input_details = interpreter.get_input_details()
    output_details = interpreter.get_output_details()

    input_data = np.expand_dims(live_features, axis=0).astype(np.float32)

    interpreter.set_tensor(input_details[0]["index"], input_data)
    interpreter.invoke()

    output_data = interpreter.get_tensor(output_details[0]["index"])

    return float(output_data[0][0])


def build_live_feature_vector(weather_data: dict, thermal_result: dict):
    """
    Convert live weather + thermal readings into the exact 13-feature model input.
    """
    fwi_result = compute_uob_fwi(weather_data)

    row = list(fwi_result["weather_vector"])
    row.append(fwi_result["fwi_score"] / 100.0)
    row.append(np.clip((thermal_result["max_temp_c"] - 20.0) / 130.0, 0.0, 1.0))
    row.append(np.clip(thermal_result["hotspot_count"] / 5.0, 0.0, 1.0))
    row.append(np.clip(thermal_result["largest_hotspot_area"] / 50.0, 0.0, 1.0))
    row.append(np.clip(thermal_result["hotspot_fraction"] / 0.15, 0.0, 1.0))
    row.append(np.clip(thermal_result["ir_confidence"], 0.0, 1.0))

    delta = thermal_result["max_temp_c"] - thermal_result["mean_temp_c"]
    row.append(np.clip(delta / 100.0, 0.0, 1.0))

    live_features = np.array(row, dtype=np.float32)

    if len(live_features) != N_FEATURES:
        raise ValueError(
            f"Feature mismatch: expected {N_FEATURES} features, "
            f"got {len(live_features)}."
        )

    return live_features, fwi_result


def run_firewatch_pipeline(weather_data: dict, ir_frame: np.ndarray = None) -> dict:
    """
    Run one complete FireWatch detection cycle.

    Runtime model:
      - TensorFlow Lite interpreter

    Cell 13 will pass:
      - WeatherLink weather
      - real 24x32 thermal frame from Pico/GY-MCU90640
    """
    global RUNTIME_MODEL, RUNTIME_THRESHOLD

    if RUNTIME_MODEL is None or RUNTIME_THRESHOLD is None:
        load_runtime_inference_assets()

    if ir_frame is None:
        raise ValueError(
            "ir_frame is required. The API flow must pass the real 24x32 frame."
        )

    thermal_result = analyse_ir_frame_mlx90640(ir_frame)
    live_features, fwi_result = build_live_feature_vector(weather_data, thermal_result)

    nn_probability = predict_with_tflite(RUNTIME_MODEL, live_features)

    temp_score = np.clip((thermal_result["max_temp_c"] - 33.0) / 33.0, 0.0, 1.0)
    area_score = np.clip(thermal_result["largest_hotspot_area"] / 12.0, 0.0, 1.0)
    hot_pixel_score = np.clip(thermal_result["hot_pixel_count"] / 12.0, 0.0, 1.0)
    confidence_score = np.clip(thermal_result["ir_confidence"], 0.0, 1.0)

    thermal_camera_score = float(np.clip(
        (temp_score * 0.40)
        + (area_score * 0.25)
        + (hot_pixel_score * 0.20)
        + (confidence_score * 0.15),
        0.0,
        1.0
    ))

    thermal_override = (
        (
            thermal_result["max_temp_c"] >= THERMAL_CRITICAL_MAX_C
            and thermal_result["largest_hotspot_area"] >= MIN_HOTSPOT_AREA
        )
        or (
            thermal_result["max_temp_c"] >= 50.0
            and thermal_result["hot_pixel_count"] >= 2
        )
        or (
            thermal_camera_score >= 0.70
            and thermal_result["hot_pixel_count"] >= 2
        )
    )

    CAMERA_WEIGHT = 0.75
    MODEL_WEIGHT = 0.25

    weighted_probability = (
        thermal_camera_score * CAMERA_WEIGHT
        + nn_probability * MODEL_WEIGHT
    )

    final_probability = float(np.clip(
        max(weighted_probability, 0.85) if thermal_override else weighted_probability,
        0.0,
        1.0
    ))

    fire_detected = (
        final_probability >= RUNTIME_THRESHOLD
        or thermal_override
    )

    if thermal_override or final_probability >= 0.75:
        alert_level = "CRITICAL"
    elif final_probability >= 0.55:
        alert_level = "HIGH"
    elif final_probability >= 0.35:
        alert_level = "MODERATE"
    else:
        alert_level = "LOW"

    return {
        "timestamp": datetime.now().isoformat(),
        "weather_data": weather_data,
        "fwi_score": fwi_result["fwi_score"],
        "fwi_class": fwi_class(fwi_result["fwi_score"]),
        "compound_weather_flag": fwi_result["compound_flag"],
        "weather_contributions": fwi_result["contributions"],
        "thermal": thermal_result,
        "thermal_analysis": thermal_result,
        "nn_probability": round(nn_probability, 4),
        "thermal_override": bool(thermal_override),
        "final_probability": round(final_probability, 4),
        "fire_detected": bool(fire_detected),
        "alert_level": alert_level,
        "predicted_behavior": predict_fire_behavior(
            weather_data,
            fwi_result["fwi_score"]
        ),
        "runtime_threshold": round(float(RUNTIME_THRESHOLD), 4),
        "model_name": RUNTIME_METADATA.get(
            "model_name",
            "UOB_Sensor_Fusion_TFLite_v2"
        ),
    }

In [ ]:
# ===========================================================================
# CELL 9B — Force-create/load TFLite runtime assets and verify
# ===========================================================================

from pathlib import Path

print("Before loading:")
print("Metadata exists:", Path(MODEL_METADATA_PATH).exists(), MODEL_METADATA_PATH)
print("Keras checkpoint exists:", Path(CKPT_PATH).exists(), CKPT_PATH)
print("TFLite exists:", Path(TFLITE_MODEL_PATH).exists(), TFLITE_MODEL_PATH)

print("\nLoading runtime inference assets now...")
load_runtime_inference_assets()

print("\nAfter loading:")
print("Metadata exists:", Path(MODEL_METADATA_PATH).exists(), MODEL_METADATA_PATH)
print("Keras checkpoint exists:", Path(CKPT_PATH).exists(), CKPT_PATH)
print("TFLite exists:", Path(TFLITE_MODEL_PATH).exists(), TFLITE_MODEL_PATH)
print("Runtime model loaded:", RUNTIME_MODEL is not None)
print("Runtime threshold loaded:", RUNTIME_THRESHOLD is not None, RUNTIME_THRESHOLD)

Before loading:
Metadata exists: True /content/firewatch_deploy/model_metadata_v2.json
Keras checkpoint exists: True /content/firewatch_checkpoints/best_model_v2.keras
TFLite exists: False /content/firewatch_checkpoints/firewatch_model_v2.tflite

Loading runtime inference assets now...
TensorFlow Lite model not found. Creating it from Keras checkpoint...
Saved artifact at '/tmp/tmplaratxmq'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 13), dtype=tf.float32, name='sensor_fusion_input')
Output Type:
  TensorSpec(shape=(None, 1), dtype=tf.float32, name=None)
Captures:
  133086302628816: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133083621309328: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133083621306256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133083621309712: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133083621307024: TensorSpec(shape=(), dtype=tf.resource, name=None)
  133083621

In [ ]:
# ===========================================================================
# CELL 10 — Geospatial Evacuation Routing Engine
# ===========================================================================
# This engine performs all geospatial computation automatically.
# It does NOT manually invent routes. It:
#   1. Downloads the real road network from OpenStreetMap (via OSMnx)
#   2. Builds a fire hazard polygon from fire origin + spread direction
#   3. Selects or generates 3 safe zone polygons
#   4. Runs shortest-path routing on the actual road graph
#   5. Avoids roads that pass through the fire hazard zone
#   6. Ranks 3 distinct routes as priority 1 (safest), 2, 3

# UOB campus approximate centroid (Koura, North Lebanon)
UOB_LAT = 34.3733
UOB_LON = 35.8317

# Search radius for the road network (metres)
ROAD_NETWORK_RADIUS_M = 5000

# Projection used for metric calculations (UTM Zone 37N covers Lebanon)
UTM_CRS = CRS.from_epsg(32637)
WGS84_CRS = CRS.from_epsg(4326)

# Transform helpers
_to_utm    = Transformer.from_crs(WGS84_CRS, UTM_CRS, always_xy=True)
_to_wgs84  = Transformer.from_crs(UTM_CRS, WGS84_CRS, always_xy=True)


def _latlon_to_utm(lat: float, lon: float):
    """Convert WGS84 lat/lon to UTM Zone 37N (x, y) in metres."""
    x, y = _to_utm.transform(lon, lat)
    return x, y


def _utm_to_latlon(x: float, y: float):
    """Convert UTM Zone 37N (x, y) back to WGS84 lat/lon."""
    lon, lat = _to_wgs84.transform(x, y)
    return lat, lon


def build_fire_hazard_polygon(
    fire_lat: float,
    fire_lon: float,
    fire_radius_m: float,
    spread_direction_deg: float,
    fwi_score: float,
) -> Polygon:
    """
    Build a fire hazard polygon in UTM coordinates.

    The polygon is an ellipse stretched in the prevailing wind/spread direction.
    The major axis length scales with FWI score (higher danger = larger zone).

    Parameters
    ----------
    fire_lat, fire_lon   : fire origin in WGS84
    fire_radius_m        : base radius of the fire hazard zone (metres)
    spread_direction_deg : compass bearing (0=N, 90=E) fire is spreading toward
    fwi_score            : 0–100; used to scale the downwind axis length

    Returns
    -------
    Shapely Polygon in UTM coordinates (EPSG:32637)
    """
    if fire_radius_m <= 0:
        fire_radius_m = 300.0

    # Scale the downwind axis by FWI (moderate = 1.5×, extreme = 3.0×)
    downwind_scale = 1.0 + (fwi_score / 100.0) * 2.0
    fire_x, fire_y = _latlon_to_utm(fire_lat, fire_lon)
    origin = Point(fire_x, fire_y)

    # Create base circle, then stretch it in the spread direction
    base_circle = origin.buffer(fire_radius_m)

    # Rotation angle: Shapely uses counter-clockwise from east;
    # compass bearing is clockwise from north. Convert:
    rot_angle_rad = math.radians(90.0 - spread_direction_deg)

    # Build the ellipse by scaling in x (major axis) and leaving y unchanged,
    # then rotating to align with spread direction.
    # We achieve this with affine_transform on the buffered circle.
    from shapely.affinity import scale as shapely_scale, rotate as shapely_rotate
    ellipse = shapely_scale(base_circle, xfact=downwind_scale, yfact=1.0, origin=origin)
    hazard_polygon = shapely_rotate(ellipse, -spread_direction_deg, origin=origin, use_radians=False)

    return hazard_polygon


def generate_safe_zones(
    fire_lat: float,
    fire_lon: float,
    spread_direction_deg: float,
    fire_radius_m: float = 300.0,
    predefined_zones: list = None,
) -> list:
    """
    Return 3 safe zone polygons (Shapely Polygon, UTM coordinates).

    If predefined_zones are provided (list of dicts with lat/lon/radius_m),
    those are used first. Otherwise, fallback zones are generated
    at 3 compass directions away from the fire (upwind + two flanks).

    Parameters
    ----------
    fire_lat, fire_lon      : fire origin (WGS84)
    spread_direction_deg    : direction fire is spreading toward (compass °)
    fire_radius_m           : radius used for sizing the safe zone buffers
    predefined_zones        : optional list of {"lat":..., "lon":..., "radius_m":...}

    Returns
    -------
    List of exactly 3 Shapely Polygons in UTM coordinates.
    """
    fire_x, fire_y = _latlon_to_utm(fire_lat, fire_lon)

    if predefined_zones and len(predefined_zones) >= 3:
        zones = []
        for z in predefined_zones[:3]:
            zx, zy = _latlon_to_utm(z["lat"], z["lon"])
            r = max(z.get("radius_m", 400.0), 100.0)
            zones.append(Point(zx, zy).buffer(r))
        return zones

    # Auto-generate safe zones in directions away from the fire spread
    # Upwind is the safest direction (opposite of spread).
    upwind_deg = (spread_direction_deg + 180.0) % 360.0
    flank1_deg = (spread_direction_deg + 120.0) % 360.0
    flank2_deg = (spread_direction_deg - 120.0) % 360.0

    safe_distance_m = max(fire_radius_m * 4.0, 1200.0)
    zone_radius_m   = max(fire_radius_m * 1.5,  400.0)

    zones = []
    for direction_deg in [upwind_deg, flank1_deg, flank2_deg]:
        angle_rad = math.radians(90.0 - direction_deg)  # Compass → math angle
        zx = fire_x + safe_distance_m * math.cos(angle_rad)
        zy = fire_y + safe_distance_m * math.sin(angle_rad)
        zones.append(Point(zx, zy).buffer(zone_radius_m))

    return zones


def _download_road_network(lat: float, lon: float, radius_m: float = ROAD_NETWORK_RADIUS_M):
    """
    Download the drivable road network around a point from OpenStreetMap.

    Returns an OSMnx graph projected into UTM Zone 37N.
    Raises RuntimeError if the download fails so the caller can activate fallback.
    """
    try:
        G_wgs = ox.graph_from_point(
            (lat, lon), dist=radius_m, network_type="drive", retain_all=False
        )
        G_utm = ox.project_graph(G_wgs, to_crs=UTM_CRS)
        return G_utm
    except Exception as exc:
        raise RuntimeError(f"OSMnx road network download failed: {exc}") from exc


def _build_fallback_grid_graph(
    fire_x: float, fire_y: float, radius_m: float = ROAD_NETWORK_RADIUS_M
) -> nx.DiGraph:
    """
    Fallback: build a synthetic rectangular grid graph when OSMnx is unavailable.

    The grid has 400 m spacing over a 10 km × 10 km bounding box centred on the fire.
    Edge weights are Euclidean distances. This guarantees routing can always proceed.

    NOTE: This fallback produces straight-line approximate routes, not real roads.
    Label the routes as FALLBACK_ROUTE in the DTO (see create_evacuation_dtos).
    """
    G = nx.DiGraph()
    step = 400
    half = int(radius_m // step)
    node_id = 0
    coord_map = {}

    for i in range(-half, half + 1):
        for j in range(-half, half + 1):
            nx_ = fire_x + i * step
            ny  = fire_y + j * step
            G.add_node(node_id, x=nx_, y=ny)
            coord_map[(i, j)] = node_id
            node_id += 1

    for i in range(-half, half + 1):
        for j in range(-half, half + 1):
            u = coord_map[(i, j)]
            for di, dj in [(0, 1), (1, 0), (0, -1), (-1, 0)]:
                ni_, nj = i + di, j + dj
                if (ni_, nj) in coord_map:
                    v = coord_map[(ni_, nj)]
                    G.add_edge(u, v, length=step)
                    G.add_edge(v, u, length=step)

    return G


def _edges_intersect_hazard(
    G, u: int, v: int, hazard_polygon: Polygon, use_osmnx: bool = True
) -> bool:
    """Return True if the edge (u, v) passes through the fire hazard polygon."""
    try:
        if use_osmnx:
            ux, uy = G.nodes[u]["x"], G.nodes[u]["y"]
            vx, vy = G.nodes[v]["x"], G.nodes[v]["y"]
        else:
            ux, uy = G.nodes[u]["x"], G.nodes[u]["y"]
            vx, vy = G.nodes[v]["x"], G.nodes[v]["y"]
        edge_line = LineString([(ux, uy), (vx, vy)])
        return hazard_polygon.intersects(edge_line)
    except Exception:
        return False


def _safe_graph(G, hazard_polygon: Polygon, use_osmnx: bool = True) -> nx.DiGraph:
    """
    Return a copy of G with all edges that cross the hazard polygon removed.
    Also removes edges whose endpoint nodes lie inside the hazard polygon.
    """
    H = G.copy()
    edges_to_remove = []
    nodes_in_hazard = set()

    for n, data in H.nodes(data=True):
        pt = Point(data["x"], data["y"])
        if hazard_polygon.contains(pt):
            nodes_in_hazard.add(n)

    for u, v, data in list(H.edges(data=True)):
        if u in nodes_in_hazard or v in nodes_in_hazard:
            edges_to_remove.append((u, v))
        elif _edges_intersect_hazard(G, u, v, hazard_polygon, use_osmnx):
            edges_to_remove.append((u, v))

    H.remove_edges_from(edges_to_remove)
    H.remove_nodes_from(nodes_in_hazard)
    return H


def _nearest_node(G, x: float, y: float) -> int:
    """Find the node in G whose (x, y) is closest to (x, y)."""
    best_node, best_dist = None, float("inf")
    for n, data in G.nodes(data=True):
        dx = data["x"] - x
        dy = data["y"] - y
        d  = dx*dx + dy*dy
        if d < best_dist:
            best_dist, best_node = d, n
    return best_node


def _route_to_linestring_wkt(G, route_nodes: list) -> str:
    """Convert a list of graph node IDs into a WKT LINESTRING in WGS84."""
    coords_wgs = []
    for n in route_nodes:
        xm, ym = G.nodes[n]["x"], G.nodes[n]["y"]
        lat, lon = _utm_to_latlon(xm, ym)
        coords_wgs.append((lon, lat))  # WKT uses (lon lat) order
    if len(coords_wgs) < 2:
        return None
    line = LineString(coords_wgs)
    return swkt.dumps(line, rounding_precision=6)


def _polygon_to_wkt(polygon: Polygon, in_utm: bool = True) -> str:
    """Convert a Shapely Polygon (UTM or WGS84) to WKT POLYGON in WGS84."""
    if in_utm:
        # Convert all exterior coordinates from UTM to WGS84
        ext_coords = []
        for (xm, ym) in polygon.exterior.coords:
            lat, lon = _utm_to_latlon(xm, ym)
            ext_coords.append((lon, lat))
        poly_wgs = Polygon(ext_coords)
    else:
        poly_wgs = polygon
    return swkt.dumps(poly_wgs, rounding_precision=6)


def _route_length_m(G, route_nodes: list) -> float:
    """Sum edge lengths along a route (metres)."""
    total = 0.0
    for u, v in zip(route_nodes[:-1], route_nodes[1:]):
        edge_data = G.get_edge_data(u, v) or {}
        if isinstance(edge_data, dict):
            # OSMnx multi-edge dict: pick shortest key
            if 0 in edge_data:
                total += edge_data[0].get("length", 0.0)
            else:
                total += next(iter(edge_data.values()), {}).get("length", 0.0)
        else:
            total += edge_data.get("length", 0.0)
    return total


def _estimated_time_str(distance_km: float, speed_kmh: float = 30.0) -> str:
    """Return a human-readable estimated travel time string."""
    if distance_km <= 0:
        return "Unknown"
    minutes = (distance_km / speed_kmh) * 60.0
    if minutes < 60:
        return f"{int(round(minutes))} minutes"
    hours = int(minutes // 60)
    mins  = int(round(minutes % 60))
    return f"{hours} h {mins} min"


def _routes_are_distinct(route_a: list, route_b: list, min_diff_fraction: float = 0.3) -> bool:
    """
    Return True if two routes differ by at least min_diff_fraction of their nodes.
    Prevents returning three nearly identical paths.
    """
    set_a = set(route_a)
    set_b = set(route_b)
    overlap = len(set_a & set_b)
    smaller = min(len(set_a), len(set_b))
    if smaller == 0:
        return True
    return (overlap / smaller) < (1.0 - min_diff_fraction)


def generate_evacuation_routes(
    fire_lat: float,
    fire_lon: float,
    fire_id: str,
    fwi_score: float,
    spread_direction_deg: float,
    fire_radius_m: float = 300.0,
    predefined_safe_zones: list = None,
) -> list:
    """
    Compute 3 ranked evacuation routes from the fire location to safe zones.

    Algorithm
    ─────────
    1. Download the real road network from OSMnx (fallback: synthetic grid).
    2. Build the fire hazard polygon (directional ellipse).
    3. Remove all roads that intersect the hazard zone from the graph.
    4. Generate 3 safe zone polygons.
    5. For each safe zone, find the shortest path from the nearest unburned
       road node to the nearest node inside the safe zone.
    6. Filter duplicate routes and re-rank by path safety (fewest hazard
       crossings on alternative segments) and distance.
    7. Return route objects ready for backend insertion.

    Returns
    -------
    List of up to 3 route dicts with keys:
        route_status, route_priority, route_path, safe_zone,
        distance_km, estimated_time, fire_id
    """
    fire_x, fire_y = _latlon_to_utm(fire_lat, fire_lon)

    # ── Step 1: Road network ────────────────────────────────────────────────
    use_osmnx  = True
    is_fallback = False
    try:
        G = _download_road_network(fire_lat, fire_lon, radius_m=max(fire_radius_m * 6, ROAD_NETWORK_RADIUS_M))
        print("  Road network downloaded from OpenStreetMap.")
    except RuntimeError as e:
        print(f"  OSMnx unavailable ({e}). Using synthetic grid fallback.")
        G = _build_fallback_grid_graph(fire_x, fire_y, radius_m=ROAD_NETWORK_RADIUS_M)
        use_osmnx  = False
        is_fallback = True

    if G.number_of_nodes() == 0:
        print("  ERROR: Road graph is empty. Cannot generate routes.")
        return []

    # ── Step 2: Hazard polygon ───────────────────────────────────────────────
    hazard_polygon = build_fire_hazard_polygon(
        fire_lat, fire_lon, fire_radius_m, spread_direction_deg, fwi_score
    )

    # ── Step 3: Safe graph (roads minus hazard zone) ─────────────────────────
    G_safe = _safe_graph(G, hazard_polygon, use_osmnx=use_osmnx)

    if G_safe.number_of_nodes() < 10:
        print("  WARNING: Very few safe nodes remain after hazard removal. Routes may be limited.")

    # ── Step 4: Safe zones ───────────────────────────────────────────────────
    safe_zones = generate_safe_zones(
        fire_lat, fire_lon, spread_direction_deg, fire_radius_m, predefined_safe_zones
    )

    # ── Step 5: Origin node — nearest safe node to the fire ──────────────────
    # We start routing from just outside the fire, not from inside the hazard.
    origin_node = _nearest_node(G_safe, fire_x, fire_y)
    if origin_node is None:
        print("  ERROR: Could not find a safe origin node near the fire.")
        return []

    # ── Step 6: Route to each safe zone ─────────────────────────────────────
    candidate_routes = []
    hazard_wkt = _polygon_to_wkt(hazard_polygon, in_utm=True)

    for zone_idx, zone_polygon in enumerate(safe_zones):
        zone_centroid = zone_polygon.centroid
        dest_node     = _nearest_node(G_safe, zone_centroid.x, zone_centroid.y)
        zone_wkt      = _polygon_to_wkt(zone_polygon, in_utm=True)

        if dest_node is None or dest_node == origin_node:
            print(f"  Safe zone {zone_idx+1}: no reachable destination node. Skipping.")
            continue

        try:
            route_nodes = nx.shortest_path(G_safe, origin_node, dest_node, weight="length")
        except nx.NetworkXNoPath:
            print(f"  Safe zone {zone_idx+1}: no path found in safe graph.")
            continue
        except nx.NodeNotFound:
            print(f"  Safe zone {zone_idx+1}: node not found in safe graph.")
            continue

        if len(route_nodes) < 2:
            continue

        route_wkt    = _route_to_linestring_wkt(G_safe, route_nodes)
        if route_wkt is None:
            continue

        dist_m       = _route_length_m(G_safe, route_nodes)
        dist_km      = round(dist_m / 1000.0, 3)
        est_time     = _estimated_time_str(dist_km)

        candidate_routes.append({
            "_nodes":       route_nodes,
            "_dist_km":     dist_km,
            "_zone_idx":    zone_idx,
            "route_status": "FALLBACK_ROUTE" if is_fallback else "COMPUTED",
            "route_path":   route_wkt,
            "safe_zone":    zone_wkt,
            "distance_km":  dist_km,
            "estimated_time": est_time,
            "fire_id":      fire_id,
        })

    # ── Step 7: Deduplicate, rank, and package ───────────────────────────────
    # Sort by distance (shortest first); then filter out near-duplicates.
    candidate_routes.sort(key=lambda r: r["_dist_km"])

    final_routes = []
    for route in candidate_routes:
        if len(final_routes) >= 3:
            break
        # Check it is distinct from all already selected routes
        is_dup = any(
            not _routes_are_distinct(route["_nodes"], prev["_nodes"])
            for prev in final_routes
        )
        if not is_dup:
            final_routes.append(route)

    # Assign priorities and remove internal fields
    dtos = []
    used_zone_indices = set()

    for priority, route in enumerate(final_routes, start=1):
        used_zone_indices.add(route.get("_zone_idx"))

        dto = {
            "route_status":   route["route_status"],
            "route_priority": priority,
            "route_path":     route["route_path"],
            "safe_zone":      route["safe_zone"],
            "distance_km":    route["distance_km"],
            "estimated_time": route["estimated_time"],
            "fire_id":        route["fire_id"],
        }
        dtos.append(dto)

    # ------------------------------------------------------------------
    # Demo/project guarantee:
    # If the real road graph only produced 1–2 routes, still return 3
    # safe-zone options using direct fallback lines to the remaining zones.
    # The frontend can later use OSRM from the resident location to the
    # selected safe zone, so these are still useful common evacuation options.
    # ------------------------------------------------------------------
    if len(dtos) < 3:
        used_priorities = {route["route_priority"] for route in dtos}
        next_priority = 1

        for zone_idx, zone_polygon in enumerate(safe_zones):
            if len(dtos) >= 3:
                break

            # Do not duplicate a safe zone that already has a computed route.
            if zone_idx in used_zone_indices:
                continue

            while next_priority in used_priorities:
                next_priority += 1

            zone_centroid = zone_polygon.centroid
            safe_lat, safe_lon = _utm_to_latlon(zone_centroid.x, zone_centroid.y)

            fallback_line = LineString([
                (fire_lon, fire_lat),
                (safe_lon, safe_lat),
            ])

            zone_wkt = _polygon_to_wkt(zone_polygon, in_utm=True)
            route_wkt = swkt.dumps(fallback_line, rounding_precision=6)

            distance_km = round(
                math.sqrt(
                    (zone_centroid.x - fire_x) ** 2
                    + (zone_centroid.y - fire_y) ** 2
                ) / 1000.0,
                3
            )

            dtos.append({
                "route_status": "FALLBACK_ROUTE",
                "route_priority": next_priority,
                "route_path": route_wkt,
                "safe_zone": zone_wkt,
                "distance_km": distance_km,
                "estimated_time": _estimated_time_str(distance_km),
                "fire_id": fire_id,
            })

            used_priorities.add(next_priority)
            used_zone_indices.add(zone_idx)
            next_priority += 1

    if not dtos:
        print("  WARNING: No valid evacuation routes could be computed.")

    return dtos


def send_fire_risk_prediction_to_backend(
    backend_graphql_url,
    zone_location,
    risk_level,
    fire_id=None,
):
    mutation = """
    mutation PublishFireRiskPrediction($input: FireRiskPredictionInput!) {
      publishFireRiskPrediction(input: $input) {
        success
        message
      }
    }
    """

    payload = {
        "query": mutation,
        "variables": {
            "input": {
                "zone_location": zone_location,
                "risk_level": risk_level,
                "fire_id": fire_id,
            }
        },
    }

    res = requests.post(backend_graphql_url, json=payload, timeout=30)
    print("Backend status:", res.status_code)

    try:
        body = res.json()
    except Exception:
        print("Backend returned non-JSON response:")
        print(res.text)
        return None

    print(body)

    if res.status_code != 200 or "errors" in body:
        print("Backend GraphQL error:")
        return None

    return body

In [ ]:
# ===========================================================================
# CELL 11 — WeatherLink live data fetch
# ===========================================================================

WEATHERLINK_TIMEOUT_SEC = 15
WEATHER_FETCH_RETRIES = 3

WEATHERLINK_URL = (
    "https://www.weatherlink.com/embeddablePage/getData/"
    "80f6b8d1b7cc4791b2ec9245ef25f522"
)

_last_good_weather = {
    "humidity": 45.0,
    "rainfall": 0.0,
    "temperature": 26.0,
    "wind_speed": 10.0,
    "pressure": 1013.0,
    "wind_dir": 180.0,
}


def _to_float_or_default(value, default: float) -> float:
    try:
        if value is None:
            return float(default)

        if isinstance(value, str):
            cleaned = (
                value.replace("°", "")
                .replace("%", "")
                .replace("mm", "")
                .replace("km/h", "")
                .replace("hPa", "")
                .replace(",", "")
                .strip()
            )
            return float(cleaned)

        return float(value)

    except Exception:
        return float(default)


def _find_first_value(data, keys: list[str]):
    """
    Recursively search a nested WeatherLink JSON payload for the first matching key.
    This makes the parser safer if WeatherLink changes nesting.
    """
    if isinstance(data, dict):
        for key in keys:
            if key in data:
                return data[key]

        for value in data.values():
            found = _find_first_value(value, keys)
            if found is not None:
                return found

    elif isinstance(data, list):
        for item in data:
            found = _find_first_value(item, keys)
            if found is not None:
                return found

    return None


def _validate_weather_dict(weather: dict) -> dict:
    """
    Enforce the exact weather keys expected by the FWI and fusion pipeline.
    """
    return {
        "humidity": _to_float_or_default(weather.get("humidity"), 45.0),
        "rainfall": _to_float_or_default(weather.get("rainfall"), 0.0),
        "temperature": _to_float_or_default(weather.get("temperature"), 26.0),
        "wind_speed": _to_float_or_default(weather.get("wind_speed"), 10.0),
        "pressure": _to_float_or_default(weather.get("pressure"), 1013.0),
        "wind_dir": _to_float_or_default(weather.get("wind_dir"), 180.0),
    }


def _extract_weatherlink_weather(data: dict) -> dict:
    """
    Extract EshMagan weather fields from WeatherLink JSON.
    Supports both flat and nested payloads.
    """
    weather = {
        "humidity": _find_first_value(data, [
            "humidity",
            "hum",
            "relative_humidity",
            "relativeHumidity",
        ]),
        "rainfall": _find_first_value(data, [
            "rainfall",
            "rain",
            "rain_rate",
            "rainfall_daily",
            "rain_day",
            "daily_rain",
        ]),
        "temperature": _find_first_value(data, [
            "temperature",
            "temp",
            "temp_c",
            "outside_temperature",
            "outsideTemp",
        ]),
        "wind_speed": _find_first_value(data, [
            "wind_speed",
            "windSpeed",
            "wind_speed_avg",
            "wind_avg",
            "wind",
        ]),
        "pressure": _find_first_value(data, [
            "pressure",
            "barometer",
            "bar",
            "bar_absolute",
            "bar_sea_level",
        ]),
        "wind_dir": _find_first_value(data, [
            "wind_dir",
            "windDirection",
            "wind_direction",
            "wind_degrees",
            "wind_dir_deg",
        ]),
    }

    return _validate_weather_dict(weather)


def get_live_weather_data() -> dict:
    """
    Fetch live weather data from WeatherLink.

    Returns the last known good packet only when WeatherLink fails.
    """
    global _last_good_weather

    last_error = None

    for attempt in range(1, WEATHER_FETCH_RETRIES + 1):
        try:
            response = requests.get(
                WEATHERLINK_URL,
                timeout=WEATHERLINK_TIMEOUT_SEC,
                headers={
                    "User-Agent": "EshMagan/2",
                    "Accept": "application/json,text/plain,*/*",
                },
            )
            response.raise_for_status()

            data = response.json()
            weather = _extract_weatherlink_weather(data)

            _last_good_weather = weather

            print("WeatherLink live weather fetched:", weather)
            return weather

        except Exception as error:
            last_error = error
            print(
                f"WeatherLink fetch attempt {attempt}/{WEATHER_FETCH_RETRIES} failed: {error}"
            )
            time.sleep(1.2)

    fallback = _validate_weather_dict(_last_good_weather)
    print(f"Using last known good WeatherLink packet after fetch failure: {last_error}")
    print("Weather fallback packet:", fallback)
    return fallback

In [ ]:
# ===========================================================================
# Runtime backend URL
# ===========================================================================

BACKEND_GRAPHQL_URL = "https://broadband-pelvis-jigsaw.ngrok-free.dev/eshmagan"

In [ ]:
# =============================================================================
# CELL 12 — FireLab fire-danger prediction notifications
# =============================================================================
#
#
# FireLab source note:
# The PDF source uses the word "Mohafazat" in section headers.
# EshMagan code/output uses "Governorate".
# =============================================================================


# ---------------------------------------------------------------------------
# FireLab PDF URL templates
# ---------------------------------------------------------------------------

FIRELAB_PDF_URLS = {
    "Mf": (
        "https://firelab.balamand.edu.lb/FireLabWeb/Content/PDF/Mf/"
        "Fire_Danger_ForeCast_Report_{date_str}.pdf"
    ),
    "Ecmwf": (
        "https://firelab.balamand.edu.lb/FireLabWeb/Content/PDF/Ecmwf/"
        "Fire_Danger_ForeCast_Report_{date_str}.pdf"
    ),
}


# ---------------------------------------------------------------------------
# Risk tables
# ---------------------------------------------------------------------------

FIRELAB_RISK_LABELS = {
    "NR": "No Risk",
    "VL": "Very Low",
    "L":  "Low",
    "M":  "Moderate",
    "H":  "High",
    "VH": "Very High",
    "E":  "Extreme",
}

FIRELAB_NOTIFY_CODES = {"H", "VH", "E"}

FIRELAB_BACKEND_LEVEL = {
    "H":  "HIGH",
    "VH": "HIGH",
    "E":  "CRITICAL",
}


# ---------------------------------------------------------------------------
# Default scope
# ---------------------------------------------------------------------------

FIRELAB_TARGET_GOVERNORATES: list[str] = ["North Lebanon"]

FIRELAB_POLL_INTERVAL_SEC: int = 3 * 3600

_FIRELAB_DEFAULT_TARGETS = object()


# ---------------------------------------------------------------------------
# Governorate centroids for backend notification zone_location
# POINT format is longitude latitude.
# ---------------------------------------------------------------------------

GOVERNORATE_CENTROIDS: dict[str, str] = {
    "North Lebanon":  f"POINT({UOB_LON} {UOB_LAT})",
    "Akkar":          "POINT(36.1500 34.5500)",
    "Baalbek-Hermel": "POINT(36.2167 34.0000)",
    "Bekaa":          "POINT(35.9020 33.8462)",
    "Mount Lebanon":  "POINT(35.7500 33.8333)",
    "Beirut":         "POINT(35.5017 33.8886)",
    "South Lebanon":  "POINT(35.3667 33.2667)",
    "Nabatieh":       "POINT(35.4833 33.3792)",
}


# ---------------------------------------------------------------------------
# Runtime deduplication
# ---------------------------------------------------------------------------

_firelab_notified: set[tuple] = set()


# ---------------------------------------------------------------------------
# Regex helpers
# ---------------------------------------------------------------------------

_RISK = re.compile(r"^(NR|VL|VH|[LMHE])$")

# Keep "Mohafazat" here because this is the source PDF wording.
_SECTION = re.compile(
    r"Mohafazat\s+([\w][\w\s\-]*?)\s+Forecast",
    re.IGNORECASE,
)

_CAZA = re.compile(
    r"Caza:\s*([\w][\w\s\-]*)",
    re.IGNORECASE,
)


# ---------------------------------------------------------------------------
# Backend URL resolver
# ---------------------------------------------------------------------------

def _resolve_firelab_backend_url(backend_graphql_url: str | None = None) -> str:
    """
    Resolve backend URL for FireLab notifications.

    Prefer an explicit function argument.
    Fall back to global BACKEND_GRAPHQL_URL when Cell 13 or a runtime config
    defines it.
    """
    if backend_graphql_url:
        return backend_graphql_url

    global_url = globals().get("BACKEND_GRAPHQL_URL")

    if global_url:
        return global_url

    raise ValueError(
        "Missing backend_graphql_url. Pass it directly or define BACKEND_GRAPHQL_URL before running FireLab notifications."
    )


# ---------------------------------------------------------------------------
# Fetch
# ---------------------------------------------------------------------------

def fetch_firelab_pdf(
    source: str = "Mf",
    date_str: str | None = None,
    timeout: int = 30,
) -> bytes | None:
    """
    Download a FireLab forecast PDF.

    source:
        "Mf"    = Météo-France primary report
        "Ecmwf" = ECMWF fallback report

    date_str:
        YYYYMMDD. Defaults to today's date.
    """
    if date_str is None:
        date_str = date.today().strftime("%Y%m%d")

    if source not in FIRELAB_PDF_URLS:
        raise ValueError(f"Unknown FireLab source: {source}. Use 'Mf' or 'Ecmwf'.")

    url = FIRELAB_PDF_URLS[source].format(date_str=date_str)
    print(f"[FireLab] Fetching {source} PDF → {url}")

    try:
        response = requests.get(
            url,
            timeout=timeout,
            headers={
                "User-Agent": "EshMagan/2",
                "Accept": "application/pdf,*/*",
            },
        )

        if response.status_code == 200 and response.content[:4] == b"%PDF":
            print(f"[FireLab] OK — {len(response.content):,} bytes.")
            return response.content

        print(f"[FireLab] HTTP {response.status_code} — PDF not available for {date_str}.")
        return None

    except requests.RequestException as error:
        print(f"[FireLab] Network error: {error}")
        return None


# ---------------------------------------------------------------------------
# Parse
# ---------------------------------------------------------------------------

def _parse_text_line(line: str) -> list[tuple[str, str, str, str]]:
    """
    Extract area + 3-day risk-code groups from one text line.

    Expected source pattern:
        Area Name CODE CODE CODE
    Multiple area groups may appear on the same line.
    """
    tokens = line.split()
    results: list[tuple[str, str, str, str]] = []
    i = 0

    while i < len(tokens):
        if _RISK.match(tokens[i]):
            i += 1
            continue

        name_parts: list[str] = []

        while i < len(tokens) and not _RISK.match(tokens[i]):
            name_parts.append(tokens[i])
            i += 1

        if not name_parts:
            i += 1
            continue

        codes: list[str] = []

        while i < len(tokens) and _RISK.match(tokens[i]) and len(codes) < 3:
            codes.append(tokens[i])
            i += 1

        if len(codes) == 3:
            results.append((
                " ".join(name_parts),
                codes[0],
                codes[1],
                codes[2],
            ))

    return results


def _normalise_governorate_name(name: str) -> str:
    """
    Normalize governorate names extracted from the FireLab PDF.
    """
    cleaned = " ".join(str(name or "").strip().split())
    cleaned = cleaned.title()

    aliases = {
        "North": "North Lebanon",
        "North Lebanon": "North Lebanon",
        "South": "South Lebanon",
        "South Lebanon": "South Lebanon",
        "Mount Lebanon": "Mount Lebanon",
        "Baalbek Hermel": "Baalbek-Hermel",
        "Baalbek-Hermel": "Baalbek-Hermel",
        "Beqaa": "Bekaa",
        "Bekaa": "Bekaa",
        "Nabatiyeh": "Nabatieh",
        "Nabatieh": "Nabatieh",
        "Akkar": "Akkar",
        "Beirut": "Beirut",
    }

    return aliases.get(cleaned, cleaned)


def parse_firelab_pdf(
    pdf_bytes: bytes,
    target_governorates: list[str] | None = None,
) -> list[dict]:
    """
    Extract FireLab forecast records.

    Returns one record per area:
        area, governorate, caza,
        day1_code, day2_code, day3_code,
        day1_label, day2_label, day3_label,
        day1_date, day2_date, day3_date
    """
    if not pdf_bytes:
        return []

    today = date.today()
    day_dates = [
        (today + timedelta(days=0)).isoformat(),
        (today + timedelta(days=1)).isoformat(),
        (today + timedelta(days=2)).isoformat(),
    ]

    def in_scope(governorate: str) -> bool:
        if target_governorates is None:
            return True

        return any(
            target.lower() in governorate.lower()
            for target in target_governorates
        )

    records: list[dict] = []

    current_governorate = "Unknown"
    current_caza = "Unknown"

    with pdfplumber.open(io.BytesIO(pdf_bytes)) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""

            for line in text.splitlines():
                line = line.strip()

                if not line:
                    continue

                section_match = _SECTION.search(line)
                if section_match:
                    current_governorate = _normalise_governorate_name(
                        section_match.group(1)
                    )
                    current_caza = "Unknown"
                    continue

                caza_match = _CAZA.search(line)
                if caza_match:
                    current_caza = caza_match.group(1).strip().title()
                    continue

                if not in_scope(current_governorate):
                    continue

                for area, d1, d2, d3 in _parse_text_line(line):
                    records.append({
                        "area": area,
                        "governorate": current_governorate,
                        "caza": current_caza,
                        "day1_code": d1,
                        "day2_code": d2,
                        "day3_code": d3,
                        "day1_label": FIRELAB_RISK_LABELS.get(d1, d1),
                        "day2_label": FIRELAB_RISK_LABELS.get(d2, d2),
                        "day3_label": FIRELAB_RISK_LABELS.get(d3, d3),
                        "day1_date": day_dates[0],
                        "day2_date": day_dates[1],
                        "day3_date": day_dates[2],
                    })

    scope = ", ".join(target_governorates) if target_governorates else "all Lebanon"
    print(f"[FireLab] {len(records)} areas parsed ({scope}).")

    return records


# ---------------------------------------------------------------------------
# Classify
# ---------------------------------------------------------------------------

def classify_firelab_alerts(records: list[dict]) -> list[dict]:
    """
    Return one alert per area × forecast day where risk is H, VH, or E.
    """
    alerts: list[dict] = []

    for record in records:
        for day_number in (1, 2, 3):
            code = record[f"day{day_number}_code"]

            if code not in FIRELAB_NOTIFY_CODES:
                continue

            alerts.append({
                "area": record["area"],
                "governorate": record["governorate"],
                "caza": record["caza"],
                "forecast_date": record[f"day{day_number}_date"],
                "day_number": day_number,
                "risk_code": code,
                "risk_label": FIRELAB_RISK_LABELS[code],
                "backend_risk_level": FIRELAB_BACKEND_LEVEL[code],
            })

    return alerts


# ---------------------------------------------------------------------------
# Notify
# ---------------------------------------------------------------------------

def _centroid_for_governorate(governorate: str) -> str:
    """
    Return WKT POINT for a governorate.
    Falls back to UOB/Koura.
    """
    governorate = str(governorate or "").strip()

    for key, wkt in GOVERNORATE_CENTROIDS.items():
        if key.lower() in governorate.lower() or governorate.lower() in key.lower():
            return wkt

    return f"POINT({UOB_LON} {UOB_LAT})"


def send_firelab_alerts(
    alerts: list[dict],
    backend_graphql_url: str | None = None,
    dry_run: bool = False,
) -> list[dict]:
    """
    Send FireLab prediction notifications.

    This function only calls send_fire_risk_prediction_to_backend().
    It never creates fire records and never generates routes.
    """
    global _firelab_notified

    backend_url = _resolve_firelab_backend_url(backend_graphql_url)

    results: list[dict] = []

    for alert in alerts:
        key = (
            alert["area"],
            alert["forecast_date"],
            alert["risk_code"],
        )

        if key in _firelab_notified:
            continue

        summary = (
            f"  [{alert['risk_label']:9s}]  "
            f"{alert['area']:35s}  "
            f"({alert['governorate']} / {alert['caza']})  "
            f"Day {alert['day_number']} ({alert['forecast_date']})"
        )

        if dry_run:
            print(f"[FireLab DRY-RUN] Would notify:{summary}")
            _firelab_notified.add(key)
            results.append({
                "alert": alert,
                "sent": False,
                "dry_run": True,
            })
            continue

        print(f"[FireLab] Notifying:{summary}")

        try:
            response = send_fire_risk_prediction_to_backend(
                backend_graphql_url=backend_url,
                zone_location=_centroid_for_governorate(alert["governorate"]),
                risk_level=alert["backend_risk_level"],
                fire_id=None,
            )

            _firelab_notified.add(key)
            results.append({
                "alert": alert,
                "sent": True,
                "response": response,
            })

        except Exception as error:
            print(f"[FireLab] Backend error for {alert['area']}: {error}")
            results.append({
                "alert": alert,
                "sent": False,
                "error": str(error),
            })

    return results


# ---------------------------------------------------------------------------
# Display helper
# ---------------------------------------------------------------------------

def _print_firelab_alert_table(alerts: list[dict]) -> None:
    print(
        f"  {'RISK':9}  "
        f"{'AREA':35}  "
        f"{'GOVERNORATE':20}  "
        f"{'DISTRICT':20}  "
        f"DAY  DATE"
    )
    print("  " + "─" * 105)

    for alert in alerts:
        print(
            f"  {alert['risk_label']:9}  "
            f"{alert['area']:35}  "
            f"{alert['governorate']:20}  "
            f"{alert['caza']:20}  "
            f"{alert['day_number']}    "
            f"{alert['forecast_date']}"
        )

    print()


# ---------------------------------------------------------------------------
# Single test
# ---------------------------------------------------------------------------

def run_firelab_single_test(
    backend_graphql_url: str | None = None,
    target_governorates=_FIRELAB_DEFAULT_TARGETS,
    source: str = "Mf",
    dry_run: bool = True,
) -> list[dict]:
    """
    Fetch one FireLab report, parse it, show alerts, and optionally notify backend.
    """
    if target_governorates is _FIRELAB_DEFAULT_TARGETS:
        targets = FIRELAB_TARGET_GOVERNORATES
    else:
        targets = target_governorates

    print(f"\n{'=' * 72}")
    print(
        f"  FireLab Single Test — source={source}  "
        f"targets={targets or 'All Lebanon'}  dry_run={dry_run}"
    )
    print(f"{'=' * 72}\n")

    pdf_bytes = fetch_firelab_pdf(source=source)

    if not pdf_bytes:
        print("[FireLab] PDF unavailable. Try source='Ecmwf' or check the date.")
        return []

    records = parse_firelab_pdf(
        pdf_bytes=pdf_bytes,
        target_governorates=targets,
    )

    alerts = classify_firelab_alerts(records)

    print(f"  {len(records)} areas parsed — {len(alerts)} at H/VH/E.\n")

    if alerts:
        _print_firelab_alert_table(alerts)
        send_firelab_alerts(
            alerts=alerts,
            backend_graphql_url=backend_graphql_url,
            dry_run=dry_run,
        )
    else:
        print("  No areas at High / Very High / Extreme today.")

    return alerts


# ---------------------------------------------------------------------------
# Monitoring loop
# ---------------------------------------------------------------------------

def run_firelab_monitoring_loop(
    backend_graphql_url: str | None = None,
    target_governorates=_FIRELAB_DEFAULT_TARGETS,
    source_priority: list[str] | None = None,
    poll_interval_sec: int = FIRELAB_POLL_INTERVAL_SEC,
    dry_run: bool = False,
) -> None:
    """
    Periodically fetch FireLab PDFs, parse risk, and send prediction notifications.

    Run this separately from Cell 13.
    """
    if target_governorates is _FIRELAB_DEFAULT_TARGETS:
        targets = FIRELAB_TARGET_GOVERNORATES
    else:
        targets = target_governorates

    sources = source_priority or ["Mf", "Ecmwf"]

    print("=" * 72)
    print("  FireLab Prediction Monitoring — STARTED")
    print(f"  Governorate(s) : {', '.join(targets) if targets else 'All Lebanon'}")
    print(f"  Sources        : {' → '.join(sources)}")
    print(f"  Poll interval  : {poll_interval_sec // 60} min")
    print("  Notify on      : High / Very High / Extreme")
    print(f"  Dry run        : {dry_run}")
    print("=" * 72)

    while True:
        started_at = time.time()
        print(f"\n[FireLab] {datetime.now():%Y-%m-%d %H:%M:%S} — cycle start")

        pdf_bytes = None

        for source in sources:
            pdf_bytes = fetch_firelab_pdf(source=source)
            if pdf_bytes:
                break

        if not pdf_bytes:
            print("[FireLab] No PDF from any source — will retry next cycle.")
        else:
            records = parse_firelab_pdf(
                pdf_bytes=pdf_bytes,
                target_governorates=targets,
            )

            alerts = classify_firelab_alerts(records)
            print(f"[FireLab] {len(alerts)} alert(s) at H/VH/E threshold.")

            if alerts:
                _print_firelab_alert_table(alerts)
                send_firelab_alerts(
                    alerts=alerts,
                    backend_graphql_url=backend_graphql_url,
                    dry_run=dry_run,
                )
            else:
                print("[FireLab] All areas within safe range — no notification sent.")

        sleep_for = max(0.0, poll_interval_sec - (time.time() - started_at))
        next_run = datetime.now() + timedelta(seconds=sleep_for)

        print(f"[FireLab] Next cycle: {next_run:%Y-%m-%d %H:%M:%S}")
        time.sleep(sleep_for)

In [ ]:
print("\nEshMagan v2 runtime cells loaded.")
print("Run Cell 13 to start the Pico/GY-MCU90640 API server.")


EshMagan v2 runtime cells loaded.
Run Cell 13 to start the Pico/GY-MCU90640 API server.


In [ ]:
# ===========================================================================
# CELL 13 — EshMagan Colab AI API Server Cell
# ===========================================================================

!pip install flask flask-cors -q

from flask import Flask, request, jsonify
from flask_cors import CORS
import numpy as np
import requests
import threading
import time
import json

# ----------------
# Backend settings
# ----------------

BACKEND_GRAPHQL_URL = "https://broadband-pelvis-jigsaw.ngrok-free.dev/eshmagan"

DEFAULT_FIRE_LOCATION_WKT = "POINT(35.8317 34.3733)"
FIRE_SOURCE = "Infrared"
FIRE_SEVERITY_LEVEL = 5

CONFIRMATION_FRAMES = 3
TRIGGER_COOLDOWN_SECONDS = 0

last_backend_trigger_time = 0
confirmation_votes = 0


# ----------------
# Demo weather input
# ----------------
# This is sent into your notebook AI pipeline.
# You can adjust these values later.

DEFAULT_WEATHER_DATA = {
    "humidity": 35,
    "rainfall": 0,
    "temperature": 30,
    "wind_speed": 10,
    "pressure": 1010,
    "wind_dir": 180
}

backend_trigger_running = False
last_backend_result = {
    "triggered": False,
    "reason": "not_triggered_yet",
}

def wkt_point_to_lat_lon(point_wkt):
    """
    Converts WKT POINT(lng lat) into lat/lon.
    Backend stores WKT as POINT(longitude latitude).
    """
    cleaned = point_wkt.strip().replace("POINT(", "").replace(")", "")
    lng_text, lat_text = cleaned.split()
    return float(lat_text), float(lng_text)


def build_common_evacuation_routes_for_fire(ai_result, fire_location_wkt):
    """
    Generates common fire-level evacuation routes and safe zones.
    These are NOT per-resident routes.

    Resident frontend will later use OSRM from resident location
    to the selected safe zone.
    """
    if "generate_evacuation_routes" not in globals():
        print("generate_evacuation_routes is not defined. No routes generated.", flush=True)
        return []

    fire_lat, fire_lon = wkt_point_to_lat_lon(fire_location_wkt)

    predicted_behavior = ai_result.get("predicted_behavior", {}) or {}
    spread_deg = float(predicted_behavior.get("wind_direction_deg", 180.0))

    fwi_score = float(ai_result.get("fwi_score", 50.0))

    thermal = ai_result.get("thermal_analysis") or ai_result.get("thermal") or {}
    max_temp = float(thermal.get("max_temp_c", 55.0))

    # Demo-friendly dynamic radius.
    # Keep this moderate because frontend uses this fire-level route/safe zone,
    # not a per-user evacuation path.
    fire_radius_m = max(300.0, min(900.0, 300.0 + max(0.0, max_temp - 55.0) * 8.0))

    print("Generating common AI evacuation routes...", flush=True)
    print("Fire lat/lon:", fire_lat, fire_lon, flush=True)
    print("Spread direction:", spread_deg, flush=True)
    print("FWI score:", fwi_score, flush=True)
    print("Fire radius:", fire_radius_m, flush=True)

    routes = generate_evacuation_routes(
        fire_lat=fire_lat,
        fire_lon=fire_lon,
        fire_id="AI_PENDING_FIRE",
        fwi_score=fwi_score,
        spread_direction_deg=spread_deg,
        fire_radius_m=fire_radius_m,
    )

    cleaned_routes = []

    for route in routes:
        cleaned_routes.append({
            # Use a safe DB/UI status.
            # Do NOT send "COMPUTED" or "FALLBACK_ROUTE" if your DB expects Open/Closed/etc.
            "route_status": "Open",
            "route_priority": int(route.get("route_priority", 1)),
            "route_path": route["route_path"],
            "safe_zone": route["safe_zone"],
            "distance_km": float(route.get("distance_km", 0)),
            "estimated_time": str(route.get("estimated_time", "Unknown")),
        })

    print(f"Common evacuation routes generated: {len(cleaned_routes)}", flush=True)
    return cleaned_routes

# ----------------
# Backend trigger
# ----------------

def send_fire_to_backend_from_ai(ai_result, sensor_info=None):
    global last_backend_trigger_time

    now = time.time()

    if sensor_info is None:
        sensor_info = {}

    fire_location_wkt = sensor_info.get("location_wkt") or DEFAULT_FIRE_LOCATION_WKT

    # IMPORTANT:
    # Keep this as a DB-allowed value.
    # Do not use sensor label because your database rejected it before.
    fire_source = FIRE_SOURCE

    sensor_label = sensor_info.get("label") or "Unknown sensor"

    print("Sending fire to backend at:", fire_location_wkt, flush=True)
    print("Fire source:", fire_source, flush=True)
    print("Sensor label:", sensor_label, flush=True)

    if now - last_backend_trigger_time < TRIGGER_COOLDOWN_SECONDS:
        return {
            "triggered": False,
            "reason": "cooldown_active"
        }

    evacuation_routes = []

    try:
        evacuation_routes = build_common_evacuation_routes_for_fire(
            ai_result=ai_result,
            fire_location_wkt=fire_location_wkt,
        )
    except Exception as route_error:
        print("Common evacuation route generation failed:", str(route_error), flush=True)
        evacuation_routes = []

    mutation = """
    mutation CreateFireAndTriggerSystem($input: CreateFireInput!) {
      createFireAndTriggerSystem(input: $input) {
        fire_id
        fire_source
        fire_location
        fire_severitylevel
        is_extinguished
        is_verified
        created_at
        updated_at
      }
    }
    """

    variables = {
        "input": {
            "fire_source": fire_source,
            "fire_location": fire_location_wkt,
            "fire_severitylevel": FIRE_SEVERITY_LEVEL,
            "is_extinguished": False,
            "is_verified": True,
            "evacuation_routes": evacuation_routes
        }
    }

    try:
        response = requests.post(
            BACKEND_GRAPHQL_URL,
            json={
                "query": mutation,
                "variables": variables
            },
            timeout=45
        )

        payload = response.json()

        print("Backend HTTP status:", response.status_code, flush=True)
        print("Backend raw response:", response.text[:2000], flush=True)
        print("Backend JSON payload:", json.dumps(payload, indent=2), flush=True)

        data = payload.get("data") or {}
        errors = payload.get("errors") or []

        created_fire = data.get("createFireAndTriggerSystem")

        if response.status_code == 200 and created_fire:
            last_backend_trigger_time = now

            return {
                "triggered": True,
                "reason": "backend_trigger_success",
                "backend_response": created_fire,
                "evacuation_route_count": len(evacuation_routes),
                "evacuation_routes_generated": len(evacuation_routes) > 0
            }

        return {
            "triggered": False,
            "reason": "backend_graphql_error" if errors else "backend_error",
            "backend_response": payload,
            "evacuation_route_count": len(evacuation_routes),
            "evacuation_routes_generated": len(evacuation_routes) > 0,
            "error": json.dumps(errors, indent=2)[:700] if errors else response.text[:700]
        }

    except Exception as error:
        print("BACKEND EXCEPTION:", str(error), flush=True)

        return {
            "triggered": False,
            "reason": "backend_exception",
            "evacuation_route_count": len(evacuation_routes),
            "evacuation_routes_generated": len(evacuation_routes) > 0,
            "error": str(error)
        }


# ----------------
# AI adapter
# ----------------

def run_colab_ai_on_frame(frame):
    """
    Receives a 24x32 NumPy thermal frame.
    Calls the notebook AI pipeline if available.

    Weather priority:
      1. Live WeatherLink data
      2. DEFAULT_WEATHER_DATA only if WeatherLink unexpectedly fails
      3. Thermal-only fallback if full AI pipeline fails
    """

    if "run_firewatch_pipeline" in globals():
        try:
            try:
                weather_data = get_live_weather_data()
                weather_source = "WeatherLink"
            except Exception as weather_error:
                print(
                    "WeatherLink failed unexpectedly. Using DEFAULT_WEATHER_DATA:",
                    str(weather_error),
                    flush=True
                )
                weather_data = DEFAULT_WEATHER_DATA
                weather_source = "DEFAULT_WEATHER_DATA"

            try:
                result = run_firewatch_pipeline(
                    weather_data=weather_data,
                    ir_frame=frame
                )
            except TypeError:
                result = run_firewatch_pipeline(
                    weather_data,
                    frame
                )

            # Normalize notebook output for the local dashboard/API response.
            if "thermal" in result and "thermal_analysis" not in result:
                result["thermal_analysis"] = result["thermal"]

            result["weather_source"] = weather_source
            result["weather_data"] = weather_data

            return result

        except Exception as error:
            print("Full AI pipeline failed. Falling back to thermal-only mode:", str(error), flush=True)

    if "analyse_ir_frame_mlx90640" in globals():
        thermal = analyse_ir_frame_mlx90640(frame)

        max_temp = float(thermal.get("max_temp_c", np.max(frame)))
        mean_temp = float(thermal.get("mean_temp_c", np.mean(frame)))
        confidence = float(thermal.get("ir_confidence", 0.0))
        largest_area = int(thermal.get("largest_hotspot_area", 0))
        hot_count = int(thermal.get("hot_pixel_count", 0))

        fire_detected = (
            max_temp >= 43.0
            and hot_count >= 2
            and largest_area >= 2
        ) or confidence >= 0.60

        probability = max(
            confidence,
            min(1.0, max(0.0, (max_temp - 35.0) / 25.0))
        )

        if fire_detected:
            alert_level = "HIGH"
        elif max_temp >= 35.0:
            alert_level = "MODERATE"
        else:
            alert_level = "LOW"

        return {
            "fire_detected": fire_detected,
            "final_probability": probability,
            "alert_level": alert_level,
            "weather_source": "thermal_only_fallback",
            "weather_data": None,
            "thermal_analysis": {
                **thermal,
                "max_temp_c": max_temp,
                "mean_temp_c": mean_temp,
                "hot_pixel_count": hot_count,
                "largest_hotspot_area": largest_area,
                "ir_confidence": confidence
            }
        }

    max_temp = float(np.max(frame))
    mean_temp = float(np.mean(frame))
    hot_pixels = int(np.sum(frame >= 43.0))

    fire_detected = max_temp >= 43.0 and hot_pixels >= 2

    return {
        "fire_detected": fire_detected,
        "final_probability": 0.85 if fire_detected else 0.05,
        "alert_level": "HIGH" if fire_detected else "LOW",
        "weather_source": "basic_thermal_fallback",
        "weather_data": None,
        "thermal_analysis": {
            "max_temp_c": max_temp,
            "mean_temp_c": mean_temp,
            "hot_pixel_count": hot_pixels,
            "largest_hotspot_area": hot_pixels,
            "ir_confidence": 0.85 if fire_detected else 0.0
        }
    }

def run_backend_trigger_async(ai_result, sensor_info):
    global backend_trigger_running, last_backend_result

    try:
        backend_trigger_running = True
        result = send_fire_to_backend_from_ai(ai_result, sensor_info)
        last_backend_result = result
        print("Async backend trigger result:", result, flush=True)

    except Exception as error:
        last_backend_result = {
            "triggered": False,
            "reason": "backend_async_exception",
            "error": str(error),
        }
        print("Async backend trigger failed:", str(error), flush=True)

    finally:
        backend_trigger_running = False

# ----------------
# Flask API server
# ----------------

app = Flask(__name__)
CORS(app)


@app.route("/health", methods=["GET"])
def health():
    return jsonify({
        "status": "ok",
        "message": "EshMagan Colab AI server is running"
    })


@app.route("/analyze_frame", methods=["POST"])
def analyze_frame():
    global confirmation_votes

    try:
        payload = request.get_json()

        if payload is None:
            return jsonify({
                "ok": False,
                "error": "Missing JSON body"
            }), 400

        pixels = payload.get("pixels")
        seq = payload.get("seq")
        local_stats = payload.get("stats", {})
        sensor_info = payload.get("sensor", {})

        if pixels is None:
            return jsonify({
                "ok": False,
                "error": "Missing pixels"
            }), 400

        if len(pixels) != 768:
            return jsonify({
                "ok": False,
                "error": f"Expected 768 pixels, got {len(pixels)}"
            }), 400

        frame = np.array(pixels, dtype=np.float32).reshape((24, 32))

        ai_result = run_colab_ai_on_frame(frame)

        fire_detected = bool(ai_result.get("fire_detected", False))
        probability = float(ai_result.get("final_probability", 0.0))
        alert_level = ai_result.get("alert_level", "LOW")

        if fire_detected:
            confirmation_votes += 1
        else:
            confirmation_votes = 0

        backend_result = {
            "triggered": False,
            "reason": "not_confirmed"
        }

        status = "NORMAL"

        if fire_detected and confirmation_votes < CONFIRMATION_FRAMES:
            status = "FIRE_CANDIDATE"

        if fire_detected and confirmation_votes >= CONFIRMATION_FRAMES:
            status = "FIRE_CONFIRMED"
            if not backend_trigger_running:
                threading.Thread(
                    target=run_backend_trigger_async,
                    args=(ai_result.copy(), sensor_info.copy()),
                    daemon=True,
                ).start()

                backend_result = {
                    "triggered": False,
                    "reason": "backend_trigger_started_async",
                }
            else:
                backend_result = {
                    "triggered": False,
                    "reason": "backend_trigger_already_running",
                }

        thermal = ai_result.get("thermal_analysis", {})

        return jsonify({
            "ok": True,
            "seq": seq,
            "status": status,
            "fire_detected": fire_detected,
            "confirmation_votes": confirmation_votes,
            "confirmation_required": CONFIRMATION_FRAMES,
            "final_probability": probability,
            "alert_level": alert_level,
            "thermal_analysis": thermal,
            "backend": backend_result,
            "last_backend_result": last_backend_result,
            "local_stats": local_stats,
            "sensor": sensor_info
        })

    except Exception as error:
        return jsonify({
            "ok": False,
            "error": str(error)
        }), 500


# ---------------------------------------------------------------------------
# READ-ONLY INCIDENT CONTEXT ENDPOINT
# ---------------------------------------------------------------------------
# Used by IncidentDetailsScreen.
# This endpoint does NOT create fires.
# This endpoint does NOT increment confirmation votes.
# This endpoint does NOT call createFireAndTriggerSystem.

@app.route("/incident_context", methods=["POST"])
def incident_context():
    try:
        payload = request.get_json(force=True) or {}

        fire_id = payload.get("fire_id")
        fire_location = payload.get("fire_location") or {}
        fire_severitylevel = payload.get("fire_severitylevel")

        lat = fire_location.get("lat") or fire_location.get("latitude") or UOB_LAT
        lon = fire_location.get("lng") or fire_location.get("lon") or fire_location.get("longitude") or UOB_LON

        lat = float(lat)
        lon = float(lon)

        weather = get_live_weather_data()
        fwi_result = compute_uob_fwi(weather)
        behavior = predict_fire_behavior(weather, fwi_result["fwi_score"])

        model_name = "UOB_Sensor_Fusion_TFLite_v2"
        runtime_threshold = None

        try:
            if RUNTIME_MODEL is None or RUNTIME_THRESHOLD is None:
                load_runtime_inference_assets()

            model_name = RUNTIME_METADATA.get("model_name", model_name)
            runtime_threshold = float(RUNTIME_THRESHOLD)
        except Exception as model_error:
            print("[Incident Context] Model metadata unavailable:", str(model_error), flush=True)

        return jsonify({
            "ok": True,
            "mode": "incident_context_read_only",
            "fire_id": fire_id,
            "fire_location": {
                "lat": lat,
                "lng": lon,
                "wkt": f"POINT({lon} {lat})",
            },
            "weather_data": weather,
            "weather_source": "WeatherLink",
            "fwi_score": fwi_result["fwi_score"],
            "fwi_class": fwi_class(fwi_result["fwi_score"]),
            "compound_weather_flag": fwi_result["compound_flag"],
            "weather_contributions": fwi_result["contributions"],
            "predicted_behavior": behavior,
            "runtime_threshold": runtime_threshold,
            "model_name": model_name,
            "fire_severitylevel": fire_severitylevel,
            "note": "Read-only incident context. No fire creation, no evacuation generation, no confirmation votes.",
        })

    except Exception as error:
        return jsonify({
            "ok": False,
            "error": str(error),
        }), 500


# ---------------------------------------------------------------------------
# READ-ONLY FIRELAB STATUS ENDPOINT
# ---------------------------------------------------------------------------
# Used by IncidentDetailsScreen FireLabWidget.
# This endpoint parses the FireLab PDF and returns risk codes.
# It does NOT send backend notifications.

@app.route("/firelab_status", methods=["GET"])
def firelab_status():
    try:
        governorate = request.args.get("governorate") or "North Lebanon"
        requested_date = request.args.get("date") or date.today().strftime("%Y%m%d")

        source_priority = ["Mf", "Ecmwf"]
        selected_pdf_bytes = None
        selected_source = None
        selected_date_str = None

        start_date = datetime.strptime(requested_date, "%Y%m%d").date()

        # Try today, then previous 7 days
        for offset in range(0, 8):
            candidate_date = start_date - timedelta(days=offset)
            candidate_date_str = candidate_date.strftime("%Y%m%d")

            for source in source_priority:
                pdf_bytes = fetch_firelab_pdf(source=source, date_str=candidate_date_str)

                if pdf_bytes:
                    selected_pdf_bytes = pdf_bytes
                    selected_source = source
                    selected_date_str = candidate_date_str
                    break

            if selected_pdf_bytes:
                break

        if not selected_pdf_bytes:
            return jsonify({
                "ok": False,
                "available": False,
                "governorate": governorate,
                "requested_date": requested_date,
                "message": "No FireLab PDF available in the last 7 days.",
            }), 404

        records = parse_firelab_pdf(
            selected_pdf_bytes,
            target_governorates=[governorate],
        )

        alerts = classify_firelab_alerts(records)

        risk_rank = {
            "NR": 0,
            "VL": 1,
            "L": 2,
            "M": 3,
            "H": 4,
            "VH": 5,
            "E": 6,
        }

        risk_labels = {
            "NR": "No Risk",
            "VL": "Very Low",
            "L": "Low",
            "M": "Moderate",
            "H": "High",
            "VH": "Very High",
            "E": "Extreme",
        }

        summary = {
            "day1_code": None,
            "day2_code": None,
            "day3_code": None,
            "day1_label": None,
            "day2_label": None,
            "day3_label": None,
            "day1_date": None,
            "day2_date": None,
            "day3_date": None,
        }

        for day_number in [1, 2, 3]:
            best_code = None
            best_record = None

            for record in records:
                code = record.get(f"day{day_number}_code")

                if not code:
                    continue

                if best_code is None or risk_rank.get(code, -1) > risk_rank.get(best_code, -1):
                    best_code = code
                    best_record = record

            if best_code and best_record:
                summary[f"day{day_number}_code"] = best_code
                summary[f"day{day_number}_label"] = risk_labels.get(best_code, best_code)
                summary[f"day{day_number}_date"] = best_record.get(f"day{day_number}_date")

        return jsonify({
            "ok": True,
            "available": True,
            "governorate": governorate,
            "requested_date": requested_date,
            "report_date": selected_date_str,
            "source": selected_source,
            "is_latest_fallback": selected_date_str != requested_date,
            **summary,
            "areas": records,
            "high_risk_alerts": alerts,
        })

    except Exception as error:
        return jsonify({
            "ok": False,
            "error": str(error),
        }), 500

# ---------------------------------------------------------------------------
# BACKGROUND FIRELAB PREDICTION MONITORING
# ---------------------------------------------------------------------------

firelab_monitoring_thread = None
firelab_monitoring_started = False


def start_firelab_monitoring_background():
    """
    Start FireLab prediction monitoring in a background thread.

    This runs separately from the MLX /analyze_frame API.
    It sends possible fire threat notifications only.
    It does NOT create fires or evacuation routes.
    """
    global firelab_monitoring_thread, firelab_monitoring_started

    if firelab_monitoring_started:
        print("[FireLab] Monitoring already running in background.", flush=True)
        return

    firelab_monitoring_thread = threading.Thread(
        target=run_firelab_monitoring_loop,
        kwargs={
            "backend_graphql_url": BACKEND_GRAPHQL_URL,
            "target_governorates": FIRELAB_TARGET_GOVERNORATES,
            "source_priority": ["Mf", "Ecmwf"],
            "poll_interval_sec": FIRELAB_POLL_INTERVAL_SEC,
            "dry_run": False,
        },
        daemon=True,
    )

    firelab_monitoring_thread.start()
    firelab_monitoring_started = True

    print("[FireLab] Prediction monitoring started in background.", flush=True)


start_firelab_monitoring_background()

# ---------------------------------------------------------------------------
# Start public Colab endpoint using Cloudflare Tunnel
# ---------------------------------------------------------------------------

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess
import re
import time
import threading


def run_app():
    app.run(host="0.0.0.0", port=5001)


thread = threading.Thread(target=run_app)
thread.daemon = True
thread.start()

time.sleep(2)

cloudflared_process = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:5001"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None

print("Starting Cloudflare tunnel...")

for line in cloudflared_process.stdout:
    print(line.strip())

    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)

    if match:
        public_url = match.group(0)
        break

if public_url is None:
    print("Could not find Cloudflare public URL. Rerun this cell.")
else:
    print("===================================")
    print("Colab AI server started.")
    print("Public URL:")
    print(public_url)
    print("")
    print("Copy this into your local thermal dashboard:")
    print(public_url + "/analyze_frame")
    print("===================================")

========================================================================[FireLab] Prediction monitoring started in background.

  FireLab Prediction Monitoring — STARTED
  Governorate(s) : North Lebanon
  Sources        : Mf → Ecmwf
  Poll interval  : 180 min
  Notify on      : High / Very High / Extreme
  Dry run        : False

[FireLab] 2026-05-12 18:44:28 — cycle start
[FireLab] Fetching Mf PDF → https://firelab.balamand.edu.lb/FireLabWeb/Content/PDF/Mf/Fire_Danger_ForeCast_Report_20260512.pdf
 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on all addresses (0.0.0.0)
 * Running on http://127.0.0.1:5001
 * Running on http://172.28.0.12:5001
INFO:werkzeug:Press CTRL+C to quit


Starting Cloudflare tunnel...
2026-05-12T18:44:31Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-05-12T18:44:31Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-05-12T18:44:35Z INF +--------------------------------------------------------------------------------------------+
2026-05-12T18:44:35Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-05-12T18:44:35Z INF |  https://references-flam